# Task 4.0. Xây dựng và kiểm tra dữ liệu đồ thị đã xử lý

| | |
|---|---|
| **Mục đích** | Chuyển graph JSON của Task 3.3 thành tensor cho huấn luyện GNN, theo `docs/4_0_data_interface_spec.md` (v1.2) |
| **Đầu vào** | Kaggle Dataset `DiverseVul 9077 - Joern Code Graphs` (có `graph_a` và `graph_b`) |
| **Đầu ra** | Thư mục `processed/`: tensor theo protocol và split, vocabulary, manifest, thống kê, `validation_report.json`, bộ pilot 30 mẫu |
| **Máy chạy** | CPU, bật Internet (để `git clone` mã nguồn) |

**Sau khi chạy:** Save Version (Save & Run All), rồi New Dataset từ output với tên `DiverseVul 9077 - Processed Graph Data`. Kết quả cuối cell kiểm tra phải là `ĐẠT`: 7.799 mẫu hợp lệ, 554 graph A và 997 graph B quá lớn theo báo cáo 3.3.

In [ ]:
import subprocess
import sys
from pathlib import Path

GIT_BRANCH = "main"
REPO_URL = "https://github.com/meo225/software-bug-detection-using-graphs.git"
REPO_DIR = Path("/tmp/g41a/software-bug-detection-using-graphs")  # Ngoài thư mục output
if not Path("/kaggle/input").is_dir():
    raise RuntimeError("Notebook này chỉ chạy trên Kaggle.")
REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--branch", GIT_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "pull", "origin", GIT_BRANCH], cwd=REPO_DIR, check=True)
print(subprocess.run(["git", "log", "-1", "--oneline"], cwd=REPO_DIR, text=True, capture_output=True).stdout)

## 1. Cấu hình và tải mã nguồn

Cell trên clone nhánh `main` vào `/tmp` (ngoài thư mục output). Cell dưới xác định thư mục dữ liệu đầu vào.

In [ ]:
roots = [p.parent for p in Path("/kaggle/input").rglob("graph_a") if p.is_dir() and (p.parent / "graph_b").is_dir()]
assert roots, "Không thấy graph_a và graph_b, hãy Add Input dataset DiverseVul 9077 - Joern Code Graphs."
GRAPH_ROOT = roots[0]
OUT_DIR = Path("/kaggle/working/processed")
print("graph root:", GRAPH_ROOT)
print("graph_a:", sum(1 for _ in (GRAPH_ROOT / "graph_a").glob("*.json")), "tệp;",
      "graph_b:", sum(1 for _ in (GRAPH_ROOT / "graph_b").glob("*.json")), "tệp")

## 2. Dựng tensor và chạy kiểm tra tự động

Vocabulary và độ dài token K chỉ lấy từ tập train của từng protocol. Sau khi dựng, các tệp `.pt` được đọc lại để kiểm tra.

In [ ]:
def run(cmd):
    proc = subprocess.Popen(cmd, cwd=REPO_DIR, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    if proc.returncode:
        raise RuntimeError(f"Lệnh thất bại với mã {proc.returncode}")

run([sys.executable, "scripts/build_processed_data.py", "--graph-root", str(GRAPH_ROOT), "--out-dir", str(OUT_DIR),
     "--expected-valid", "7799", "--expected-oversized-a", "554", "--expected-oversized-b", "997"])

## 3. Tóm tắt kết quả

Số mẫu, K, vocabulary, thống kê kích thước, số mẫu có input mô hình trùng, kết quả pilot và danh sách tệp đầu ra.

In [ ]:
import json
import torch

meta = json.loads((OUT_DIR / "metadata.json").read_text(encoding="utf-8"))
print("Mẫu hợp lệ:", meta["n_valid_samples"], "| bị bỏ:", meta["skipped"])
for protocol, info in meta["protocols"].items():
    print(protocol, "K =", info["k"], "| P95 thô =", info["train_token_len_p95_raw"],
          "| vocab token =", info["n_tokens"], "| vocab loại node =", info["n_node_types"])
    for name, s in info["splits"].items():
        print("   ", name, s)
sample = torch.load(OUT_DIR / "project_wise" / "graph_b_train.pt")[0]
print({k: (tuple(v.shape) if hasattr(v, "shape") else v) for k, v in sample.items()})
print(sorted(p.name for p in OUT_DIR.rglob("*") if p.is_file()))
report = json.loads((OUT_DIR / "validation_report.json").read_text(encoding="utf-8"))
print("\nKẾT QUẢ KIỂM TRA:", "ĐẠT" if report["passed"] else "KHÔNG ĐẠT")
print("Pilot:", report.get("pilot"))
for protocol, info in report["protocols"].items():
    print(protocol, "số mẫu:", info["counts"])
    print("  Node tại giới hạn K (train):", info["stats"]["train"]["nodes_at_token_limit_k"])
    print("  Node mỗi graph (train):", info["stats"]["train"]["nodes_per_graph"])
    print("  Cạnh Graph B (train):", info["stats"]["train"]["edges_graph_b"])
    for kind, dup in info["model_input_duplicates"].items():
        print("  Trùng input", kind, dup["held_out"], "| trong nội bộ train:", dup["train_duplicates_within_train"])
